# Complete Guide to Program Parser

The Program Parser is a powerful tool for analyzing, extracting, and validating code. It provides capabilities to:
- Parse source code and extract structured components
- Identify functions, classes, and code blocks
- Validate syntax errors before execution
- Reconstruct code from parsed representations

## Table of Contents
1. [CodeParser Class](#1-CodeParser-Class)
2. [Parsing Result Classes](#2-Parsing-Result-Classes)
3. [Usage Examples](#3-Usage-Examples)
4. [Code Extraction Techniques](#4-Code-Extraction-Techniques)
5. [Advanced Validation](#5-Advanced-Validation)

## 1. CodeParser Class

The `CodeParser` is the main entry point for parsing source code. It currently supports **Python** and provides a unified interface for analyzing code structure.

In [ ]:
from algodisco.toolkit.program_parser import CodeParser, has_syntax_error

# Create a parser instance for Python
# Supported language: 'python'
parser = CodeParser("python")

print(f"Language: {parser.language_name}")

### Key Features of CodeParser

- **Language-aware parsing**: Automatically handles Python syntax specifics
- **Syntax validation**: Checks for errors before returning results
- **Structured extraction**: Breaks code into Functions, Classes, and CodeBlocks
- **Code reconstruction**: Can rebuild the original code from parsed elements

### CodeParser API

| Method | Description |
|--------|-------------|
| `parse(code)` | Parses source code and returns a `Program` object |
| `language_name` | Returns the supported language name |

## 2. Parsing Result Classes

The parser returns a hierarchical structure with four main classes:

In [ ]:
# Parsing result classes:
# - CodeBlock: Non-function/class code snippets (imports, variable assignments, etc.)
# - Function: Function definitions (including async functions)
# - Class: Class definitions with their methods and body elements
# - Program: The entire program containing all extracted elements

from algodisco.toolkit.program_parser import CodeBlock, Function, Class, Program

print("=== CodeBlock ===")
print("Attributes: code, indent")
print("Represents: Any code that is not a function or class definition")

print("\n=== Function ===")
print("Attributes: name, pre_name, post_name, body, footer, docstring, is_async")
print("Represents: Function definitions, including async def")

print("\n=== Class ===")
print("Attributes: name, pre_name, post_name, footer, body, docstring")
print("Represents: Class definitions with nested methods and body elements")

print("\n=== Program ===")
print("Attributes: scripts, functions, classes, elements")
print("Represents: The entire parsed program")

### Class Details

#### CodeBlock
Represents non-function/class code such as:
- Import statements (`import os`, `from typing import List`)
- Variable assignments (`x = 5`)
- Comments
- Any standalone expressions

#### Function
Represents a function definition with:
- `name`: Function name
- `pre_name`: Decorators and annotations before the function
- `post_name`: Return type annotations after the parentheses
- `body`: The function's internal code
- `footer`: Code after the body (rarely used)
- `docstring`: Documentation string (triple-quoted)
- `is_async`: Boolean indicating if it's an async function

#### Class
Represents a class definition with:
- `name`: Class name
- `pre_name`: Decorators and base class annotations
- `post_name`: Return type annotations (rare)
- `body`: List of elements inside the class (Methods are Functions, nested Classes, or CodeBlocks)
- `footer`: Code after the class body
- `docstring`: Class documentation

#### Program
The root of the parse tree:
- `scripts`: List of top-level CodeBlock elements
- `functions`: List of top-level Function elements
- `classes`: List of top-level Class elements
- `elements`: Combined list of all elements in parse order

## 3. Usage Examples

In [ ]:
# Parse sample code

code = """
import os

def add(a, b):
    '''Add two numbers'''
    return a + b

class Calculator:
    def __init__(self):
        self.result = 0
    
    def add(self, x):
        self.result += x
        return self.result
"""

parser = CodeParser("python")
program = parser.parse(code)

print(f"Parsing Results:")
print(f"  Number of functions: {len(program.functions)}")
print(f"  Number of classes: {len(program.classes)}")
print(f"  Number of script blocks: {len(program.scripts)}")
print(f"  Total elements: {len(program.elements)}")

In [ ]:
# Access function details

for func in program.functions:
    print(f"\nFunction: {func.name}")
    print(f"  is_async: {func.is_async}")
    print(f"  docstring: {func.docstring}")
    print(f"  pre_name (decorators): {repr(func.pre_name)}")
    print(f"  body (first 50 chars): {func.body[:50]}...")

In [ ]:
# Access class details and nested methods

for cls in program.classes:
    print(f"\nClass: {cls.name}")
    print(f"  docstring: {repr(cls.docstring)}")
    print(f"  Number of body elements: {len(cls.body)}")
    
    for item in cls.body:
        if isinstance(item, Function):
            print(f"    - Method: {item.name}")
        elif isinstance(item, Class):
            print(f"    - Nested Class: {item.name}")
        elif isinstance(item, CodeBlock):
            print(f"    - Code Block: {item.code[:30].strip()}...")

In [ ]:
# Reconstruct code from parsed program

reconstructed = str(program)
print("Reconstructed Code:")
print(reconstructed)

### Program Class Internals

The `Program` class provides several internal properties and methods for programmatic access:

In [ ]:
# Program internal structure exploration

# All elements in order of appearance
print("Elements in parse order:")
for i, elem in enumerate(program.elements):
    elem_type = type(elem).__name__
    if hasattr(elem, 'name'):
        print(f"  {i}: {elem_type} - {elem.name}")
    else:
        print(f"  {i}: {elem_type} - {elem.code[:40].strip()}...")

In [ ]:
# Program as a tree structure

def print_program_tree(program, indent=0):
    """Recursively print the program structure as a tree."""
    prefix = "  " * indent
    
    # Print top-level scripts
    for script in program.scripts:
        print(f"{prefix}Script: {script.code[:50].strip()}...")
    
    # Print top-level functions
    for func in program.functions:
        print(f"{prefix}Function: {func.name}")
    
    # Print top-level classes
    for cls in program.classes:
        print(f"{prefix}Class: {cls.name}")
        for item in cls.body:
            if isinstance(item, Function):
                print(f"{prefix}  └── Method: {item.name}")
            elif isinstance(item, Class):
                print(f"{prefix}  └── Nested Class: {item.name}")
            elif isinstance(item, CodeBlock):
                print(f"{prefix}  └── CodeBlock: {item.code[:30].strip()}...")

print("Program Tree Structure:")
print_program_tree(program)

## 4. Syntax Checking

The parser provides syntax validation to catch errors before code execution. This is crucial when processing LLM-generated code.

In [ ]:
# Syntax error checking

# Valid code example
valid_code = "def test():\n    pass"
print(f"Valid code has_syntax_error: {has_syntax_error(valid_code, 'python')}")

# Invalid code example (missing closing parenthesis)
invalid_code = "def test(:\n    pass"
print(f"Invalid code has_syntax_error: {has_syntax_error(invalid_code, 'python')}")

In [ ]:
# Automatic syntax checking during parsing

try:
    program = parser.parse("def broken(: pass")
except RuntimeError as e:
    print(f"Caught syntax error: {e}")

### Advanced Syntax Validation

For production use, always validate LLM-generated code before execution:

In [ ]:
# Comprehensive syntax validation function

def validate_code_syntax(code: str, language: str = "python") -> dict:
    """
    Validate code and return detailed results.
    
    Returns:
        dict with keys: is_valid, error_message, error_line, error_column
    """
    if has_syntax_error(code, language):
        # Try to extract more error details
        try:
            compile(code, '<string>', 'exec')
        except SyntaxError as e:
            return {
                'is_valid': False,
                'error_message': str(e),
                'error_line': e.lineno,
                'error_column': e.offset
            }
    return {'is_valid': True, 'error_message': None, 'error_line': None, 'error_column': None}

# Test with various error types
test_cases = [
    ("def foo(): pass", "valid function"),
    ("def bar(: pass", "missing closing paren"),
    ("class Foo { def bar(self): pass }", "invalid class syntax (curly braces)"),
    ("x = =", "invalid assignment"),
]

for code, description in test_cases:
    result = validate_code_syntax(code)
    print(f"{description}: valid={result['is_valid']}, error={result['error_message']}")

## 5. Code Extraction Techniques

When working with LLM responses, code is often wrapped in markdown code blocks. These techniques help extract and validate the actual code.

In [ ]:
# Technique 1: Extract code from LLM responses

from algodisco.toolkit.program_parser.utils import (
    extract_code_from_markdown_block,
    extract_code_from_response,
)

test_responses = [
    "Here is the code:\n```python\ndef sort_list(lst):\n    return sorted(lst)\n```",
    "```\ndef hello():\n    print('world')\n```",
    "Plain code without markdown",
    "```javascript\nconst x = 1;\n```",
]

for resp in test_responses:
    markdown_only = extract_code_from_markdown_block(resp, "python")
    best_effort = extract_code_from_response(resp, "python")
    print(f"markdown block: {repr(markdown_only)}")
    print(f"best effort: {repr(best_effort)}")
    print("---")


In [ ]:
# Technique 2: Validate and clean code with the built-in utility pipeline

from algodisco.toolkit.program_parser.utils import extract_code_from_response

def validate_and_clean_code(code: str, language: str = "python") -> tuple[bool, str]:
    """
    Full pipeline: extract code from a raw model response, then validate syntax.
    """
    extracted = extract_code_from_response(code, language)
    if extracted is None:
        return False, ""

    if has_syntax_error(extracted, language):
        return False, ""

    return True, extracted

valid_code = """
```python
def test():
    pass
```
"""
ok, cleaned = validate_and_clean_code(valid_code)
print(f"Valid code - ok: {ok}, code: {repr(cleaned)}")

invalid_code = """
```python
def test(:
    pass
```
"""
ok, cleaned = validate_and_clean_code(invalid_code)
print(f"Invalid code - ok: {ok}, code: {repr(cleaned)}")


In [ ]:
# Technique 3: Extract specific patterns (functions, classes)

def extract_named_patterns(code: str, language: str = "python") -> dict:
    """
    Extract code patterns like function and class definitions.
    
    Returns:
        Dictionary with lists of function and class names
    """
    # Use the parser to do the heavy lifting
    parser = CodeParser(language)
    program = parser.parse(code)
    
    return {
        'functions': [f.name for f in program.functions],
        'classes': [c.name for c in program.classes],
        'all_elements': len(program.elements)
    }

# Example: extracting from a larger code snippet
complex_code = """
import json
from typing import List, Dict

def process_data(data: List[Dict]) -> List[Dict]:
    results = []
    for item in data:
        results.append({'processed': True, **item})
    return results

class DataProcessor:
    def __init__(self, config: Dict):
        self.config = config
    
    def process(self, data: List[Dict]) -> List[Dict]:
        return process_data(data)
"""

patterns = extract_named_patterns(complex_code)
print(f"Extracted patterns: {patterns}")

In [ ]:
# Technique 4: Handle incomplete or partial code

def is_complete_code(code: str, language: str = "python") -> bool:
    """
    Check if code appears to be a complete, parseable unit.
    
    Note: This is a heuristic check. For full validation, use parse().
    """
    # Remove comments and whitespace
    code = re.sub(r'#.*$', '', code, flags=re.MULTILINE)
    code = code.strip()
    
    # Check for basic structural completeness
    # Not perfect, but catches common issues
    if not code:
        return False
    
    # Try to parse
    try:
        parser = CodeParser(language)
        parser.parse(code)
        return True
    except RuntimeError:
        return False

# Test partial vs complete code
partial = "def foo():"
complete = "def foo():\n    pass"

print(f"Partial code 'def foo():' complete: {is_complete_code(partial)}")
print(f"Complete code 'def foo(): pass' complete: {is_complete_code(complete)}")

## 6. Advanced Program Operations

Beyond basic parsing, the Program class supports various operations for code analysis and transformation.

In [ ]:
# Filter and search elements

# Find all functions with a specific name pattern
def find_functions(program: Program, pattern: str) -> list:
    """Find all functions matching a regex pattern."""
    regex = re.compile(pattern)
    return [f for f in program.functions if regex.search(f.name)]

# Find all methods in classes
def get_all_methods(program: Program) -> list:
    """Get all methods from all classes in the program."""
    methods = []
    for cls in program.classes:
        for item in cls.body:
            if isinstance(item, Function):
                methods.append((cls.name, item))
    return methods

# Example usage
functions = find_functions(program, r'^add$')
print(f"Functions named 'add': {[f.name for f in functions]}")

methods = get_all_methods(program)
print(f"All methods found: {[(cls, m.name) for cls, m in methods]}")

In [ ]:
# Program serialization and reconstruction

# The Program class can be converted to string (for code generation)
# and supports equality comparison

original = """
def foo(x):
    return x * 2
"""

parser = CodeParser("python")
prog1 = parser.parse(original)
prog2 = parser.parse(str(prog1))

# Both programs should produce identical output
print(f"Original:\n{original}")
print(f"Reconstructed:\n{str(prog2)}")
print(f"Structures match: {prog1.functions[0].name == prog2.functions[0].name}")

## Summary

| Class/Function | Description |
|---------------|-------------|
| `CodeParser` | Main parser class for parsing source code |
| `CodeParser.parse()` | Parse code and return a `Program` object |
| `has_syntax_error()` | Check if code has syntax errors |
| `CodeBlock` | Non-function/class code snippet |
| `Function` | Function definition with name, body, docstring, etc. |
| `Class` | Class definition with methods and nested elements |
| `Program` | Entire program with scripts, functions, and classes |
| `str(program)` | Reconstruct code from parsed Program |

### Best Practices

1. **Always validate LLM output**: Use `has_syntax_error()` before executing generated code
2. **Extract from markdown**: Use `extract_code_from_response()` to clean LLM responses
3. **Handle parsing errors**: Wrap `parse()` in try/except for graceful error handling
4. **Use Program for transformations**: Work with the parsed structure for analysis, not raw strings
5. **Check completeness**: For partial code, use structural checks before attempting to parse